In [1]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import torch 
from torchvision.datasets import ImageFolder

#veri setleriinin yollarını tanımlıyoruz.

nwpu_train_path = "../split_data/nwpu/train"
nwpu_val_path = "../split_data/nwpu/val"
nwpu_test_path = "../split_data/nwpu/test"

ucmerced_train_path = "../split_data/ucmerced/train"
ucmerced_val_path = "../split_data/ucmerced/val"
ucmerced_test_path = "../split_data/ucmerced/test"
###############
#NWPU veri seti ile ilgili bilgiler


#nwpu train verisi için kullanılacak transform işlemleri
from torchvision import transforms

nwpu_train_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

#nwpu test ve validasyon verisi için kullanılacak transform işlemleri
nwpu_test_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

#ucmerced train verisi için kullanılacak transform işlemleri

ucmerced_train_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

#ucmerced test,validasyon verisi için kullanılacak transform işlemleri

ucmerced_test_tranform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])


])

nwpu_train = ImageFolder(nwpu_train_path,transform=nwpu_train_tranform)
nwpu_val = ImageFolder(nwpu_val_path,transform=nwpu_test_tranform)
nwpu_test = ImageFolder(nwpu_test_path,transform=nwpu_test_tranform)
nwpu_toplam_resim_sayisi = len(nwpu_val)+ len(nwpu_train) +len(nwpu_test)
nwpu_siniflar = nwpu_train.classes

#Image Folder (görsel,sınıf) şeklinde tuple verir

ucmerced_train = ImageFolder(ucmerced_train_path,transform=ucmerced_train_tranform)
ucmerced_val = ImageFolder(ucmerced_val_path,transform=ucmerced_test_tranform)
ucmerced_test = ImageFolder(ucmerced_test_path,transform=ucmerced_test_tranform)
ucmerced_toplam_resim_sayisi = len(ucmerced_val)+ len(ucmerced_train) +len(ucmerced_test)
ucmerced_siniflar = ucmerced_train.classes

#nwpu verisetindeki train,validasyon ve test verileri için DataLoder objeleri oluşturuldu

nwpu_train_loader=DataLoader(dataset=nwpu_train,batch_size=16,shuffle=True)
nwpu_valid_loader=DataLoader(dataset=nwpu_val,batch_size=16,shuffle=False)
nwpu_test_loader=DataLoader(dataset=nwpu_test,batch_size=16,shuffle=False)

#ucmerced verisetindeki train,validasyon ve test verileri için DataLoder objeleri oluşturuldu

ucmerced_train_loader = DataLoader(dataset=ucmerced_train,batch_size=16,shuffle=True)
ucmerced_valid_loader = DataLoader(dataset=ucmerced_val,batch_size=16,shuffle=False)
ucmerced_test_loader = DataLoader(dataset=ucmerced_test,batch_size=16,shuffle=True)




##GPU kontrolü

In [2]:
import torch


#nvdia GPU sahip cihazlar için kontrol
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(device)

#mac cihazlar için GPU kontrolü
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


##Sıfırdan kendi modelimizi tanımlıcaz.TinyVGG benzeri bir model olacak ama biraz daha derin

In [3]:
from torch import nn
import torch

class kendiModelim(nn.Module):
    def __init__(self,giris_boyutu:int,ara_boyut:int,cikis_boyutu:int):
        super().__init__()

        self.blok1 = nn.Sequential(
            nn.Conv2d(
            in_channels=giris_boyutu,
            out_channels=ara_boyut,
            kernel_size=3,
            padding=1,
            stride=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=ara_boyut,
                out_channels=ara_boyut,
                kernel_size=3,
                padding=1,
                stride=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)

        )

        self.blok2 = nn.Sequential(
            nn.Conv2d(in_channels=ara_boyut,
                      out_channels=ara_boyut*2,
                      kernel_size=3,
                      padding=1,
                      stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=ara_boyut*2,
                      out_channels=ara_boyut*2,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(stride=2,kernel_size=2)
        )

        self.siniflandirici = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(p=0.1),
            nn.Linear(in_features=ara_boyut*2,out_features=cikis_boyutu)


        )

    def forward(self,X:torch.Tensor):
        return self.siniflandirici(self.blok2(self.blok1(X)))

torch.manual_seed(42)
kendiModelim_nwpu = kendiModelim(giris_boyutu=3,ara_boyut=32,cikis_boyutu=len(nwpu_train.classes))
kendiModelim_ucmerced = kendiModelim(giris_boyutu=3,ara_boyut=32,cikis_boyutu=len(ucmerced_train.classes))



In [4]:
from torchinfo import summary


summary(model=kendiModelim_nwpu,input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
kendiModelim                             [1, 45]                   --
├─Sequential: 1-1                        [1, 32, 112, 112]         --
│    └─Conv2d: 2-1                       [1, 32, 224, 224]         896
│    └─ReLU: 2-2                         [1, 32, 224, 224]         --
│    └─Conv2d: 2-3                       [1, 32, 224, 224]         9,248
│    └─ReLU: 2-4                         [1, 32, 224, 224]         --
│    └─MaxPool2d: 2-5                    [1, 32, 112, 112]         --
├─Sequential: 1-2                        [1, 64, 56, 56]           --
│    └─Conv2d: 2-6                       [1, 64, 112, 112]         18,496
│    └─ReLU: 2-7                         [1, 64, 112, 112]         --
│    └─Conv2d: 2-8                       [1, 64, 112, 112]         36,928
│    └─ReLU: 2-9                         [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-10                   [1, 64, 56, 56]           --
├─S

In [5]:
import torchvision

efficientnet_agirliklar = torchvision.models.EfficientNet_B0_Weights.DEFAULT
efficientnet_model_nwpu = torchvision.models.efficientnet_b0(weights=efficientnet_agirliklar)
efficientnet_model_ucmerced = torchvision.models.efficientnet_b0(weights=efficientnet_agirliklar)





In [6]:
from torchinfo import summary

summary(model=efficientnet_model_nwpu,
        input_size=(1,3,224,224)
        ,col_names=["input_size","trainable","num_params"],
        row_settings=["var_names"],
        depth=2,
        )


Layer (type (var_name))                                      Input Shape               Trainable                 Param #
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]          True                      --
├─Sequential (features)                                      [1, 3, 224, 224]          True                      --
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]          True                      928
│    └─Sequential (1)                                        [1, 32, 112, 112]         True                      1,448
│    └─Sequential (2)                                        [1, 16, 112, 112]         True                      16,714
│    └─Sequential (3)                                        [1, 24, 56, 56]           True                      46,640
│    └─Sequential (4)                                        [1, 40, 28, 28]           True                      242,930
│    └─Sequential (5)                             

In [7]:
import torch

for param in efficientnet_model_nwpu.features.parameters():
    param.requires_grad=False #efficientnet modelinin öğrenebilir parametrelerini kapattık(nwpu veri seti için olanı)

for param in efficientnet_model_ucmerced.features.parameters():
    param.requires_grad=False #efficientnet modelinin öğrenebilir parametrelerini kapattık(ucmerced veri seti için olanı)




efficientnet_model_nwpu.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.1),
    torch.nn.Linear(in_features=1280,out_features=len(nwpu_siniflar))
)

efficientnet_model_ucmerced.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.1),
    torch.nn.Linear(in_features=1280,out_features=len(ucmerced_siniflar))
)

from torchinfo import summary

summary(model=efficientnet_model_nwpu,
        input_size=(1,3,224,224),
        col_names=["input_size","trainable","num_params"],
        row_settings=["var_names"],
        depth=2)



Layer (type (var_name))                                      Input Shape               Trainable                 Param #
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]          Partial                   --
├─Sequential (features)                                      [1, 3, 224, 224]          False                     --
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]          False                     (928)
│    └─Sequential (1)                                        [1, 32, 112, 112]         False                     (1,448)
│    └─Sequential (2)                                        [1, 16, 112, 112]         False                     (16,714)
│    └─Sequential (3)                                        [1, 24, 56, 56]           False                     (46,640)
│    └─Sequential (4)                                        [1, 40, 28, 28]           False                     (242,930)
│    └─Sequential (5)                   

In [8]:
import torchvision

resnet_agirliklar = torchvision.models.ResNet18_Weights.DEFAULT
resnet_model_nwpu = torchvision.models.resnet18(weights=resnet_agirliklar)
resnet_model_ucmerced = torchvision.models.resnet18(weights=resnet_agirliklar)

from torchinfo import summary

summary(model=resnet_model_nwpu,
        input_size=(1,3,224,224),
        col_names=["input_size","trainable","num_params"],
        row_settings= ["var_names"],
        depth=2
        )



Layer (type (var_name))                  Input Shape               Trainable                 Param #
ResNet (ResNet)                          [1, 3, 224, 224]          True                      --
├─Conv2d (conv1)                         [1, 3, 224, 224]          True                      9,408
├─BatchNorm2d (bn1)                      [1, 64, 112, 112]         True                      128
├─ReLU (relu)                            [1, 64, 112, 112]         --                        --
├─MaxPool2d (maxpool)                    [1, 64, 112, 112]         --                        --
├─Sequential (layer1)                    [1, 64, 56, 56]           True                      --
│    └─BasicBlock (0)                    [1, 64, 56, 56]           True                      73,984
│    └─BasicBlock (1)                    [1, 64, 56, 56]           True                      73,984
├─Sequential (layer2)                    [1, 64, 56, 56]           True                      --
│    └─BasicBlock (0)  

In [9]:
import torch
from torchinfo import summary

for param in resnet_model_nwpu.parameters():
    param.requires_grad = False

for param in resnet_model_ucmerced.parameters():
    param.requires_grad = False

resnet_model_nwpu.fc = torch.nn.Linear(
    in_features=512,
    out_features=len(nwpu_siniflar)
)

resnet_model_ucmerced.fc = torch.nn.Linear(
    in_features=512,
    out_features=len(ucmerced_siniflar)
)

summary(model=resnet_model_nwpu,
        input_size=(1,3,224,224),
        col_names=["input_size","trainable","num_params"],
        row_settings=["var_names"],
        depth=2)


Layer (type (var_name))                  Input Shape               Trainable                 Param #
ResNet (ResNet)                          [1, 3, 224, 224]          Partial                   --
├─Conv2d (conv1)                         [1, 3, 224, 224]          False                     (9,408)
├─BatchNorm2d (bn1)                      [1, 64, 112, 112]         False                     (128)
├─ReLU (relu)                            [1, 64, 112, 112]         --                        --
├─MaxPool2d (maxpool)                    [1, 64, 112, 112]         --                        --
├─Sequential (layer1)                    [1, 64, 56, 56]           False                     --
│    └─BasicBlock (0)                    [1, 64, 56, 56]           False                     (73,984)
│    └─BasicBlock (1)                    [1, 64, 56, 56]           False                     (73,984)
├─Sequential (layer2)                    [1, 64, 56, 56]           False                     --
│    └─BasicBlo

In [10]:
#modellerimizi oluşturduk şimdi de modellerimizi eğitelim
#loss fonksiyonumuzu ve optimezerımızı tanımlayalım
from torch import nn 


modeller = [kendiModelim_nwpu,
            kendiModelim_ucmerced,
            efficientnet_model_nwpu,
            efficientnet_model_ucmerced,
            resnet_model_nwpu,
            resnet_model_ucmerced]


loss_fonksiyonu = nn.CrossEntropyLoss()

# 6 tane modelimiz var,her model için ayrı ayrı optimizerlar tanımlıyoruz

optimizer_kendiModel_nwpu = torch.optim.Adam(params=kendiModelim_nwpu.parameters(),lr=0.001)
optimizer_kendiModel_ucmerced = torch.optim.Adam(params=kendiModelim_ucmerced.parameters(),lr=0.001)
optimizer_efficientModel_nwpu = torch.optim.Adam(params=efficientnet_model_nwpu.parameters(),lr=0.001)
optimizer_efficientModel_ucmerced = torch.optim.Adam(params=efficientnet_model_ucmerced.parameters(),lr=0.001)
optimizer_resnetModel_nwpu = torch.optim.Adam(params=efficientnet_model_nwpu.parameters(),lr=0.001)
optimizer_resnetModel_ucmerced = torch.optim.Adam(params=resnet_model_ucmerced.parameters(),lr=0.001)



In [11]:
import torch

def train_adimi(
    model:torch.nn.Module,
    dataloader:torch.utils.data.DataLoader,
    loss_fonk:torch.nn.Module,
    optimizer:torch.optim.Optimizer):
    
    model.train()
    
    
    #başlangıçta modelin loss değeri ve doğruluğu sıfır
    train_loss = 0
    train_acc = 0
    
    
    #dataLoader (batch,x,y) şeklinde bir tuple değer döndürür
    for batch,(X,y) in(enumerate(dataloader)):
        
        y_tahmin = model(X)
        
        #tahmini y değeri ile gerçek y değeri arasındaki loss hesaplanır ve train_loss değişkenine eklenir
        loss_degeri=loss_fonk(y_tahmin,y)
        train_loss += loss_degeri.item()
        
        #önceden birikmiş gradyanlar silinir
        optimizer.zero_grad()
        
        #geri yayılım sayesinde hangi ağırlığı ne kdar değiştireceğini öğrenir
        loss_degeri.backward()
        
        #ağırlıkları günceller
        optimizer.step()
        
        #y_tahmin değeri bize her sınıf için tahmini bir logit değeri döndürür
        #logit ham (işlenmemiş) tahmin puanlarıdır
        #dim=1 her bir örnek için sınıflara ait tahmin puanlarını/olasılıklarını temsil eder
        y_tahmin_sinifi = torch.argmax(y_tahmin,dim=1)
        
        #.item() sayesinde toplam doğru sınıf sayısını tensör şeklinde normal Python'daki int sayısına çeviriyoruz
        train_acc += (y_tahmin_sinifi==y).sum().item() / len(y_tahmin)
    
    #len(dataloader) bize batch sayısını döndürür
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    
    return train_loss,train_acc


def valid_adimi(
    model:torch.nn.Module,
    dataloader:torch.utils.data.DataLoader,
    loss_fonk:torch.nn.Module):
    
    #1. Katmanları test moduna sok (Dropout vb. kapansın)
    model.eval()
    
    valid_loss = 0
    valid_acc = 0
    
    #Türev ,gradyan,geri yayılım gibi işlemleri tamamen kapat (Hızlansın)
    with torch.inference_mode():
        for batch,(X,y) in enumerate(dataloader):
            
            y_tahmin = model(X)
            
            loss_degeri = loss_fonk(y_tahmin,y)
            valid_loss+= loss_degeri.item()
            
            y_tahmin_sinifi = torch.argmax(y_tahmin,dim=1)
            valid_acc =+ (y_tahmin_sinifi==y).sum().item()/len(y)
    
    valid_loss = valid_loss/len(dataloader)
    valid_acc = valid_acc/len(dataloader)
    return valid_loss,valid_acc

def test_adimi(
    model:torch.nn.Module,
    dataloader:torch.utils.data.DataLoader,
    loss_fonk:torch.nn.Module):
    
    model.eval()
    
    test_loss = 0
    test_acc = 0
    
    with torch.inference_mode():
        
        for batch,(X,y) in enumerate(dataloader):
            
            y_tahmin = model(X)
            
            loss_degeri = loss_fonk(y_tahmin,y)
            test_loss += loss_degeri.item()
            
            y_tahmin_sinifi = torch.argmax(y_tahmin,dim=1)
            test_acc = (y_tahmin_sinifi==y).sum().item() / len(y)
            
            
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss,test_acc
            
    
    

In [ ]:
#şimdi modellerinin eğitimini başlatacak train fonksiyonumuzu yazalım.
import torch

def train(
    model:torch.nn.Module,
    train_dataLoader:torch.utils.data.DataLoader,
    valid_dataLoader:torch.utils.data.DataLoader,
    optimizer:torch.optim.Optimizer=torch.optim.adam,
    loss_fonk:torch.nn.Module = torch.nn.CrossEntropyLoss(),
    epochs:int = 10,
):
    ciktilar = {"train_acc":[],
                "train_loss":[],
                "valid_acc":[],
                "valid_loss":[]
                }
    
    for epoch in range(epochs):
        train_loss,train_acc=train_adimi(model=model,
                                        dataloader=train_dataLoader,
                                        loss_fonk=loss_fonk,
                                        optimizer=optimizer)
        
        valid_loss,valid_acc = valid_adimi(model=model,
                                           dataloader=valid_dataLoader,
                                           loss_fonk=loss_fonk)
        
        print(f"Epoch:{epoch+1} | Train Acc:{train_acc:.4f} | Train Loss:{train_loss:.4f} | Valid Acc:{valid_acc:.4f} | Valid Loss:{valid_loss:.4f}")
        
        ciktilar["train_acc"].append(train_acc)
        ciktilar["train_loss"].append(train_loss)
        ciktilar["valid_acc"].append(valid_acc)
        ciktilar["valid_loss"].append(valid_loss)
        
    return ciktilar